In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile

zip_path = "/content/drive/MyDrive/hole.zip"
extract_path = "/content/pothole"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted!")

In [ ]:
!ls /content/pothole


In [ ]:
!cat /content/pothole/data.yaml

In [ ]:
%%writefile /content/pothole/data.yaml
train: /content/pothole/train/images
val: /content/pothole/valid/images
test: /content/pothole/test/images

nc: 1
names: ['pothole']

roboflow:
  workspace: vasanthakumar662005
  project: pothole-pq3lu
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/vasanthakumar662005/pothole-pq3lu/dataset/1

In [ ]:
!cat /content/pothole/data.yaml

In [ ]:
!cut -d ' ' -f1 /content/pothole/train/labels/*.txt | sort | uniq -c

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import shutil

import os

# 2️⃣ Paths
data_yaml = '/content/pothole/data.yaml'  # your dataset
save_dir = '/content/drive/MyDrive/pothole_yolo'       # folder in Drive to save weights
os.makedirs(save_dir, exist_ok=True)

# 3️⃣ Initialize YOLOv8 model (Nano for fast training, can switch to 'yolov8s.pt')
model = YOLO('yolov8n.pt')

# 4️⃣ Train model
results = model.train( # Capture the results object which contains the Trainer
    data=data_yaml,
    epochs=50,      # adjust as needed
    imgsz=640,
    batch=16,
    device=0,       # GPU
    project=save_dir,
    name='pothole_model',
    exist_ok=True
)

# 5️⃣ After training, copy the best weights to Drive
# Use results.save_dir to get the correct path to the training run's output directory
best_weights = os.path.join(results.save_dir, 'weights', 'best.pt')
shutil.copy(best_weights, save_dir)
print(f"✅ Trained weights saved to: {save_dir}/best.pt")

# 6️⃣ Download link for local storage
from google.colab import files
files.download(os.path.join(save_dir, 'best.pt'))